# Test Visualization Script

**SHASSST, R4P50, t = 117.315250, r variable**

**Data is stored in Zarr format, pulled from DTN node using Globus. This is directly analagous to pulling from Constellation.**

For interest, you can see the data structure under /lustre/orion/proj-shared/cfd135/zarrTest/zarr/ (this is a test, some things to be optimized)
- Multiple resolutions are stored (full data, sparsed every 2 point, every 4 points etc)
- Each resolution is split into sub-cubes (~512 MB in size) so data doesn't need to all be downloaded at once.
- User specifies what indices and sparsing level they want, and script automatically downloads them the requested .npy file.
- Visualizer also provided.


**Run the steps below in order:**

1. **Authenticate with Globus** — installs dependencies, loads the libraries, and logs you in (right now, requires you to authenticate via RSA key, on Constellation would be open to anyone with Globus account)
2. **Download data** — pick a resolution level, variable and x/y/z index ranges; only the shard files your selection touches are transferred, and `zarr` reassembles the requested subdomain into a .npy file under the folder `./strata_data/`.
3. **Visualize** — choose any saved `.npy` file and slice through it.

**Before you begin:** a free [Globus account](https://www.globus.org/) is
required (sign in with Google or your institution). If the interactive widgets
don't render, run once:
`from google.colab import output; output.enable_custom_widget_manager()`

In [ ]:
#@title Step 1: Initialize packages, authenticate with Globus (may take 30 sec...)
!pip install -q "zarr>=3" globus-sdk requests ipywidgets

import glob, itertools, math, os, re

import numpy as np
import requests
import zarr
import globus_sdk
import ipywidgets as widgets
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d.art3d import Poly3DCollection
from IPython.display import display, HTML

plt.rcParams.update({"figure.dpi": 110, "font.size": 10})

# --- dataset / store configuration -------------------------------------------
# R4P50 sharded Zarr v3 store. Inner chunk is 64^3; sharded levels group inner
# chunks into larger shard files. On coarse (unsharded) levels the file
# granularity is the inner chunk itself.
LEVEL_INFO = [
    dict(label="0 - full res  16000x8000x4000",  shape=(16000, 8000, 4000), shard=(1024, 512, 256), chunk=(64, 64, 64), sub="",   available=True),
    dict(label="1 - 2x  8000x4000x2000",         shape=(8000,  4000, 2000), shard=(512,  256, 128), chunk=(64, 64, 64), sub="/1", available=True),
    dict(label="2 - 4x  4000x2000x1000",         shape=(4000,  2000, 1000), shard=(256,  128,  64), chunk=(64, 64, 64), sub="/2", available=True),
    dict(label="3 - 8x  2000x1000x500",          shape=(2000,  1000,  500), shard=None,             chunk=(64, 64, 64), sub="/3", available=False),
    dict(label="4 - 16x  1000x500x250",          shape=(1000,   500,  250), shard=None,             chunk=(64, 64, 64), sub="/4", available=False),
    dict(label="5 - 32x  500x250x125",           shape=(500,    250,  125), shard=None,             chunk=(64, 64, 64), sub="/5", available=True),
]
VARIABLE_INFO = {
    "r": "r  -  density (buoyancy)",
    "u": "u  -  streamwise velocity",
    "v": "v  -  spanwise velocity",
    "w": "w  -  vertical velocity",
}
AVAILABLE_VARS = ["r"]          # add u, v, w as those conversions finish
CMAP = "RdBu_r"

# Both persist for the life of the Colab runtime.
CACHE_ROOT     = "./strata_cache"     # fetched shards land here
SAVE_DIR       = "./strata_data"      # reconstructed .npy cubes land here
MAX_FETCH_GB   = 20                   # disk / download cap (RAM is no longer the limit: save streams to a memmap, viewing is memory-mapped)

# --- Globus HTTPS settings (per-user OAuth + HTTPS GET) --------------------
CLIENT_ID       = "d47db6dc-0428-4076-9a6e-31927d7c7704"                # registered Globus Native app id
COLLECTION_ID   = "36d521b3-c182-4071-b7d5-91db5d380d42"                # mapped collection UUID
HTTPS_BASE      = "https://m-36da98.abb646.36fe.data.globus.org"       # HTTPS-mapped collection endpoint
COLLECTION_PATH = "/lustre/orion/cfd135/proj-shared/zarrTest/zarr/"     # store root on the collection

os.makedirs(CACHE_ROOT, exist_ok=True)
os.makedirs(SAVE_DIR, exist_ok=True)
print("shard cache :", os.path.abspath(CACHE_ROOT))
print("saved cubes :", os.path.abspath(SAVE_DIR))

# --- targeted shard transfer over Globus HTTPS (inlined; no repo needed) ----
def shard_rel_paths(var, grid, sel):
    """POSIX rel paths of the metadata + data files a selection needs.

    `grid` is the shard shape (sharded level) or inner-chunk shape (coarse
    level); on-disk files live at <var>/c/<i>/<j>/<k>. Enumerating indices
    (rather than listing a directory) is what lets HTTPS GET work.
    `sel` is (x0, x1, y0, y1, z0, z1) in level-local indices.
    """
    paths = ["zarr.json", var + "/zarr.json"]
    x0, x1, y0, y1, z0, z1 = sel
    gx, gy, gz = grid
    for i in range(x0 // gx, math.ceil(x1 / gx)):
        for j in range(y0 // gy, math.ceil(y1 / gy)):
            for k in range(z0 // gz, math.ceil(z1 / gz)):
                paths.append(f"{var}/c/{i}/{j}/{k}")
    return paths


class GlobusHttpsBackend:
    """HTTPS GET of individual shard files, authorised by a Globus token."""

    def __init__(self, base_url, token):
        self.base_url = base_url.rstrip("/")
        self.token = token
        self._sess = requests.Session()

    def fetch(self, rel_paths, local_root, on_file=None):
        headers = {"Authorization": f"Bearer {self.token}"} if self.token else {}
        for rel in rel_paths:
            url = f"{self.base_url}/{rel}?download=1"
            dst = os.path.join(local_root, rel)
            r = self._sess.get(url, headers=headers, stream=True)
            if r.status_code == 404:
                if on_file is not None:
                    on_file(rel)              # count skipped shard toward progress
                continue                      # shard absent (all-fill) -> skip
            r.raise_for_status()
            os.makedirs(os.path.dirname(dst), exist_ok=True)
            with open(dst, "wb") as f:
                for chunk in r.iter_content(chunk_size=1 << 20):
                    f.write(chunk)
            if on_file is not None:
                on_file(rel)


def ensure_local(cache_root, backend, var, grid, sel, on_start=None, on_file=None):
    """Fetch the files a selection needs into cache_root (skip present shards;
    always refresh tiny metadata). Returns cache_root to open with zarr."""
    rel = shard_rel_paths(var, grid, sel)
    missing = [r for r in rel if not os.path.exists(os.path.join(cache_root, r))]
    meta = [r for r in rel if r.endswith("zarr.json")]
    need = sorted(set(missing) | set(meta))
    if on_start is not None:
        on_start(need)
    if need:
        backend.fetch(need, cache_root, on_file=on_file)
    return cache_root


# --- shared plotting helpers (used by both the download and visualize widgets)
W = widgets.Layout

_BOX_EDGES = [(0, 4), (1, 5), (2, 6), (3, 7),
              (0, 2), (1, 3), (4, 6), (5, 7),
              (0, 1), (2, 3), (4, 5), (6, 7)]


def _draw_box(ax, x0, x1, y0, y1, z0, z1, **kw):
    pts = np.array(list(itertools.product([x0, x1], [y0, y1], [z0, z1])))
    for a, b in _BOX_EDGES:
        ax.plot(*zip(pts[a], pts[b]), **kw)


# --- authenticate ------------------------------------------------------------
HTTPS_SCOPE = f"https://auth.globus.org/scopes/{COLLECTION_ID}/https"

# Force authentication with your OLCF identity (PIN + RSA SecurID) instead of
# silently reusing an existing Globus session (e.g. a Google login). Without
# this you just get a token for whatever identity is already logged in -> the
# OLCF mapped collection then rejects it with 403.
#   * prompt="login"                  -> do not reuse the current session
#   * session_required_single_domain  -> require an identity in the OLCF SSO
#                                        domain (this triggers the RSA prompt)
OLCF_DOMAIN = "sso.ccs.ornl.gov"                 # OLCF SSO domain (all OLCF users)

_client = globus_sdk.NativeAppAuthClient(CLIENT_ID)
_client.oauth2_start_flow(requested_scopes=HTTPS_SCOPE)

_qp = {"prompt": "login", "session_required_single_domain": OLCF_DOMAIN}
_authorize_url = _client.oauth2_get_authorize_url(query_params=_qp)

print("Open this URL and log in WITH YOUR OLCF IDENTITY (PIN + RSA token),")
print("then paste the code below:\n")
print(_authorize_url)

_code = input("\nAuthorisation code: ").strip()
_tok = _client.oauth2_exchange_code_for_tokens(_code)

HTTPS_TOKEN = None
for _rs, _data in _tok.by_resource_server.items():
    if _rs != "auth.globus.org":
        HTTPS_TOKEN = _data["access_token"]
        break

assert HTTPS_TOKEN, "Authentication failed: no HTTPS token received."
print("Authentication successful.")

In [ ]:
#@title Step 2: Download data
# --- backend factory, transfer + reconstruct, estimate, wireframe ----------
def _grid(info):
    """File-granularity shape: shard shape if sharded, else inner-chunk shape."""
    return info["shard"] or info["chunk"]


def make_backend(info):
    base = HTTPS_BASE.rstrip("/") + COLLECTION_PATH + info["sub"]
    return GlobusHttpsBackend(base, HTTPS_TOKEN)


def resolve_store(level_idx, var, sel, on_start=None, on_file=None):
    """Fetch just the shards this selection touches; return a store path to open."""
    info = LEVEL_INFO[level_idx]
    cache = CACHE_ROOT + info["sub"]
    ensure_local(cache, make_backend(info), var, _grid(info), sel,
                 on_start=on_start, on_file=on_file)
    return cache


def transfer_estimate(x0, x1, y0, y1, z0, z1, level_idx):
    """Return (n_shards_or_None, xfer_mb, data_mb)."""
    info = LEVEL_INFO[level_idx]
    data_mb = (x1 - x0) * (y1 - y0) * (z1 - z0) * 4 / 1e6
    if info["shard"] is None:
        return None, data_mb, data_mb
    sx, sy, sz = info["shard"]
    n = ((math.ceil(x1 / sx) - math.floor(x0 / sx)) *
         (math.ceil(y1 / sy) - math.floor(y0 / sy)) *
         (math.ceil(z1 / sz) - math.floor(z0 / sz)))
    return n, n * sx * sy * sz * 4 / 1e6, data_mb


def format_estimate(x0, x1, y0, y1, z0, z1, level_idx):
    n, xfer_mb, data_mb = transfer_estimate(x0, x1, y0, y1, z0, z1, level_idx)
    lines = []
    if n is not None:
        lines.append(f"<b>Transfer:</b> {n} shard{'s' if n != 1 else ''}  "
                     f"(~{xfer_mb / 1024:.2f} GB uncompressed)")
    lines.append(f"<b>Decompressed data:</b> {data_mb:.0f} MB")
    return "<br>".join(lines)


def draw_wireframe(level_idx, sel):
    nx, ny, nz = LEVEL_INFO[level_idx]["shape"]
    x0, x1, y0, y1, z0, z1 = sel
    fig = plt.figure(figsize=(5.2, 4.2))
    ax = fig.add_subplot(111, projection="3d")
    _draw_box(ax, 0, nx, 0, ny, 0, nz, color="0.65", lw=0.8)
    _draw_box(ax, x0, x1, y0, y1, z0, z1, color="tab:red", lw=2.5)
    ax.set_xlim(0, nx); ax.set_ylim(0, ny); ax.set_zlim(0, nz)
    ax.set_box_aspect((nx, ny, nz))
    ax.set_xlabel("x"); ax.set_ylabel("y"); ax.set_zlabel("z")
    ax.set_title("Selection within full domain", fontsize=10)
    ax.view_init(elev=22, azim=-58)
    return fig


def _aligned_blocks(a, b, step):
    """Yield [lo, hi) sub-ranges of [a, b) snapped to the shard/chunk grid,
    so each block lies within a single shard (read once, minimal RAM)."""
    lo = a
    while lo < b:
        hi = min((lo // step + 1) * step, b)
        yield lo, hi
        lo = hi


def save_cube_streaming(root, var, sel, level_idx, grid, on_start=None, on_block=None):
    """Write the selected subdomain to a .npy on disk, block by block via a
    memmap, so peak RAM is ~one shard rather than the whole cube."""
    x0, x1, y0, y1, z0, z1 = sel
    za = root[var]
    fname = f"{var}_L{level_idx}_x{x0}-{x1}_y{y0}-{y1}_z{z0}-{z1}.npy"
    path = os.path.join(SAVE_DIR, fname)
    xb = list(_aligned_blocks(x0, x1, grid[0]))
    yb = list(_aligned_blocks(y0, y1, grid[1]))
    zb = list(_aligned_blocks(z0, z1, grid[2]))
    if on_start is not None:
        on_start(len(xb) * len(yb) * len(zb))
    out = np.lib.format.open_memmap(path, mode="w+", dtype=za.dtype,
                                    shape=(x1 - x0, y1 - y0, z1 - z0))
    try:
        for a0, a1 in xb:
            for b0, b1 in yb:
                for c0, c1 in zb:
                    out[a0 - x0:a1 - x0, b0 - y0:b1 - y0, c0 - z0:c1 - z0] = \
                        za[a0:a1, b0:b1, c0:c1]
                    if on_block is not None:
                        on_block()
    finally:
        out.flush()
        del out
    return path, fname, os.path.getsize(path) / 1e6


# --- download widget -------------------------------------------------------
w_level = widgets.Dropdown(
    options=[(i["label"], k) for k, i in enumerate(LEVEL_INFO) if i["available"]],
    description="Level:", layout=W(width="480px"), style={"description_width": "60px"})
w_var = widgets.Dropdown(
    options=[(VARIABLE_INFO[v], v) for v in AVAILABLE_VARS],
    description="Variable:", layout=W(width="280px"), style={"description_width": "75px"})


def _range_slider(label, maxval, default_hi):
    return widgets.IntRangeSlider(
        value=[0, min(default_hi, maxval)], min=0, max=maxval, step=64,
        description=label, continuous_update=False,
        layout=W(width="620px"), style={"description_width": "25px"})


nx0, ny0, nz0 = LEVEL_INFO[0]["shape"]
w_x = _range_slider("x", nx0, 1024)
w_y = _range_slider("y", ny0,  512)
w_z = _range_slider("z", nz0,  256)
w_full = widgets.Checkbox(value=False, description="Download entire box (full range)",
                          indent=False, layout=W(width="340px"))
w_estimate = widgets.HTML()
w_wire     = widgets.Output()
w_fetch    = widgets.Button(description="Download", button_style="primary",
                            icon="download", layout=W(width="180px"))
w_status   = widgets.Output()


def _ranges(level_idx):
    """Read sliders; a zero-width range [i, i] becomes a single plane [i, i+1]."""
    nx, ny, nz = LEVEL_INFO[level_idx]["shape"]
    out = []
    for w, n in ((w_x, nx), (w_y, ny), (w_z, nz)):
        a, b = w.value
        if b <= a:
            a = min(a, n - 1); b = a + 1
        out += [a, b]
    return out


def _update_preview(*_):
    sel = _ranges(w_level.value)
    w_estimate.value = format_estimate(*sel, w_level.value)
    with w_wire:
        w_wire.clear_output(wait=True)
        fig = draw_wireframe(w_level.value, sel); display(fig); plt.close(fig)


def on_level_change(change):
    nx, ny, nz = LEVEL_INFO[change["new"]]["shape"]
    for w, n, d in [(w_x, nx, 1024), (w_y, ny, 512), (w_z, nz, 256)]:
        w.max = n
        w.value = [0, n] if w_full.value else [0, min(d, n)]
    _update_preview()


def on_full_change(change):
    full = change["new"]
    nx, ny, nz = LEVEL_INFO[w_level.value]["shape"]
    for w, n in ((w_x, nx), (w_y, ny), (w_z, nz)):
        w.disabled = full
        if full:
            w.value = [0, n]
    _update_preview()


def on_fetch(_):
    # grey out the button until the whole download + save finishes
    w_fetch.disabled = True
    w_fetch.button_style = ""
    try:
        sel = _ranges(w_level.value)
        x0, x1, y0, y1, z0, z1 = sel
        lvl, var = w_level.value, w_var.value

        _, _, data_mb = transfer_estimate(*sel, lvl)
        if data_mb / 1024 > MAX_FETCH_GB:
            with w_status:
                w_status.clear_output(wait=True)
                display(HTML(f"<span style='color:#b00'><b>Selection too large:</b> "
                             f"{data_mb / 1024:.1f} GB exceeds the {MAX_FETCH_GB} GB limit.<br>"
                             f"Narrow the ranges or pick a coarser level.</span>"))
            return

        # --- phase 1: download (progress bar sized in MB from the known shard size)
        grid = _grid(LEVEL_INFO[lvl])
        shard_mb = grid[0] * grid[1] * grid[2] * 4 / 1e6
        prog   = widgets.FloatProgress(value=0, min=0, max=1, bar_style="info",
                                       layout=W(width="380px"))
        plabel = widgets.HTML()
        st = {"done": 0.0, "total": 0.0}

        def _on_start(need):
            n = sum(1 for r in need if not r.endswith("zarr.json"))
            st["total"] = n * shard_mb
            prog.max = max(st["total"], 1e-9)
            if n:
                plabel.value = (f"Downloading {n} shard{'s' if n != 1 else ''} "
                                f"(~{st['total'] / 1024:.2f} GB) over Globus HTTPS…")
            else:
                plabel.value = "All shards already cached."

        def _on_file(rel):
            if rel.endswith("zarr.json"):
                return
            st["done"] += shard_mb
            prog.value = min(st["done"], prog.max)
            plabel.value = f"Downloading… {st['done']:.0f} / {st['total']:.0f} MB"

        with w_status:
            w_status.clear_output(wait=True)
            display(widgets.VBox([plabel, prog]))

        store = resolve_store(lvl, var, sel, on_start=_on_start, on_file=_on_file)

        # --- phase 2: reconstruct + save straight to a disk-backed memmap,
        #     block by block (peak RAM ~ one shard, not the whole cube)
        def _r_start(n):
            st["done"], st["total"] = 0, n
            prog.value = 0; prog.max = max(n, 1); prog.bar_style = "info"
            plabel.value = f"Reconstructing & saving… 0 / {n} blocks"

        def _r_block():
            st["done"] += 1
            prog.value = st["done"]
            plabel.value = f"Reconstructing & saving… {st['done']:.0f} / {st['total']:.0f} blocks"

        path, fname, size_mb = save_cube_streaming(
            zarr.open_group(store, mode="r"), var, sel, lvl, grid,
            on_start=_r_start, on_block=_r_block)

        with w_status:
            w_status.clear_output(wait=True)
            display(HTML(f"<b>Save complete.</b><br>"
                         f"<code>{fname}</code>&nbsp; ({size_mb:.1f} MB)"))
    finally:
        w_fetch.disabled = False
        w_fetch.button_style = "primary"


for w in (w_x, w_y, w_z):
    w.observe(_update_preview, names="value")
w_level.observe(on_level_change, names="value")
w_full.observe(on_full_change, names="value")
w_fetch.on_click(on_fetch)

panel = widgets.VBox([
    widgets.HTML("<b>Choose resolution, variable and index ranges (data will save to ./strata_data as .npy file)</b>"),
    widgets.HBox([w_level, w_var]),
    w_x, w_y, w_z, w_full,
    widgets.HTML("<hr style='margin:6px 0'>"),
    widgets.HBox([
        widgets.VBox([w_estimate, widgets.HTML("<br>"),
                      w_fetch, w_status],
                     layout=W(width="420px")),
        w_wire,
    ]),
], layout=W(padding="12px", max_width="1100px"))

_update_preview()
display(panel)

---
## Step 3 — Visualize

Reads whatever `.npy` cubes live in `./strata_data/`. Pick one, load it, and
slice through it. (Press **Refresh list** after fetching a new cube above.)

In [ ]:
#@title Step 3: Visualize
# --- single-slice viewer helpers -------------------------------------------
# orientation -> (perpendicular axis letter, index into data.shape)
PERP_AXIS = {"xy": ("z", 2), "yz": ("x", 0), "xz": ("y", 1)}


def parse_selection(fname):
    """Recover (var, level_idx, sel) from a saved filename; None if unmatched.
    sel is (x0, x1, y0, y1, z0, z1) in the level's own (global) indices."""
    base = os.path.basename(fname)
    m = re.match(r"(?P<var>[A-Za-z0-9]+)_L(?P<lvl>\d+)"
                 r"_x(?P<x0>\d+)-(?P<x1>\d+)_y(?P<y0>\d+)-(?P<y1>\d+)"
                 r"_z(?P<z0>\d+)-(?P<z1>\d+)", base)
    if not m:
        return None
    sel = (int(m["x0"]), int(m["x1"]), int(m["y0"]),
           int(m["y1"]), int(m["z0"]), int(m["z1"]))
    return m["var"], int(m["lvl"]), sel


def plot_single_slice(data, sel, var, orient, pos, vlim):
    """2D slice, axes labelled in the original (global) indices via extent."""
    x0, x1, y0, y1, z0, z1 = sel
    lo, hi = vlim
    kw = dict(cmap=CMAP, vmin=lo, vmax=hi, origin="lower", aspect="equal")
    label = VARIABLE_INFO.get(var, var)
    fig, ax = plt.subplots(figsize=(8, 5))
    if orient == "xy":
        im = ax.imshow(data[:, :, pos].T, extent=[x0, x1, y0, y1], **kw)
        ax.set(xlabel="x", ylabel="y",
               title=f"{label}  -  xy plane at z = {z0 + pos}")
    elif orient == "yz":
        im = ax.imshow(data[pos, :, :].T, extent=[y0, y1, z0, z1], **kw)
        ax.set(xlabel="y", ylabel="z",
               title=f"{label}  -  yz plane at x = {x0 + pos}")
    else:
        im = ax.imshow(data[:, pos, :].T, extent=[x0, x1, z0, z1], **kw)
        ax.set(xlabel="x", ylabel="z",
               title=f"{label}  -  xz plane at y = {y0 + pos}")
    plt.colorbar(im, ax=ax, label=label); plt.tight_layout(); plt.show()


def draw_domain_box(domain_shape, sel):
    """Full domain wireframe with a red box marking where the subcube sits.
    Drawn in the original (global) indices."""
    nx, ny, nz = domain_shape
    x0, x1, y0, y1, z0, z1 = sel
    fig = plt.figure(figsize=(4.6, 3.8))
    ax = fig.add_subplot(111, projection="3d")
    _draw_box(ax, 0, nx, 0, ny, 0, nz, color="0.65", lw=0.8)
    _draw_box(ax, x0, x1, y0, y1, z0, z1, color="tab:red", lw=2.5)
    ax.set_xlim(0, nx); ax.set_ylim(0, ny); ax.set_zlim(0, nz)
    ax.set_box_aspect((nx, ny, nz))
    ax.set_xlabel("x"); ax.set_ylabel("y"); ax.set_zlabel("z")
    ax.set_title("Subcube location in full domain", fontsize=9)
    ax.view_init(elev=22, azim=-58)
    return fig


def draw_slice_wireframe(sel, orient, pos):
    """Subcube wireframe with the red slice plane, in the original indices."""
    x0, x1, y0, y1, z0, z1 = sel
    if orient == "xy":
        z = z0 + pos; corners = [(x0, y0, z), (x1, y0, z), (x1, y1, z), (x0, y1, z)]
    elif orient == "yz":
        x = x0 + pos; corners = [(x, y0, z0), (x, y1, z0), (x, y1, z1), (x, y0, z1)]
    else:
        y = y0 + pos; corners = [(x0, y, z0), (x1, y, z0), (x1, y, z1), (x0, y, z1)]
    fig = plt.figure(figsize=(4.6, 3.8))
    ax = fig.add_subplot(111, projection="3d")
    _draw_box(ax, x0, x1, y0, y1, z0, z1, color="0.6", lw=0.9)
    ax.add_collection3d(Poly3DCollection([corners], alpha=0.35,
                        facecolor="tab:red", edgecolor="tab:red"))
    ax.set_xlim(x0, x1); ax.set_ylim(y0, y1); ax.set_zlim(z0, z1)
    ax.set_box_aspect((x1 - x0, y1 - y0, z1 - z0))
    ax.set_xlabel("x"); ax.set_ylabel("y"); ax.set_zlabel("z")
    ax.set_title("Slice position in subcube", fontsize=9)
    ax.view_init(elev=22, azim=-58)
    return fig


# --- visualizer widget -----------------------------------------------------
VSTATE = {"data": None, "sel": (0, 1, 0, 1, 0, 1), "var": "",
          "domain": (1, 1, 1), "vlim": (0, 1), "busy": False}

vw_file    = widgets.Dropdown(description="File:", layout=W(width="620px"),
                              style={"description_width": "40px"})
vw_refresh = widgets.Button(description="Refresh list", icon="refresh", layout=W(width="140px"))
vw_load    = widgets.Button(description="Load", button_style="primary", layout=W(width="100px"))
vw_orient  = widgets.ToggleButtons(options=[("xy slice", "xy"), ("yz slice", "yz"),
                                            ("xz slice", "xz")], value="xy",
                                   description="Plane:", style={"description_width": "45px"})
vw_pos     = widgets.IntSlider(value=0, min=0, max=1, description="z",
                               continuous_update=False, layout=W(width="520px"),
                               style={"description_width": "30px"})
vw_controls = widgets.VBox([vw_orient, vw_pos])
vw_status  = widgets.Output()
vw_domain  = widgets.Output()
vw_wire    = widgets.Output()
vw_img     = widgets.Output()


def _list_saved():
    return sorted(glob.glob(os.path.join(SAVE_DIR, "*.npy")))


def on_refresh(_=None):
    files = _list_saved()
    vw_file.options = [(os.path.basename(f), f) for f in files]
    with vw_status:
        vw_status.clear_output(wait=True)
        print(f"{len(files)} saved cube(s) in {SAVE_DIR}" if files
              else f"No .npy files in {SAVE_DIR} yet - run a fetch in Step 2.")


def _render(*_):
    """Redraw the slice image + slice-position wireframe (domain box is static)."""
    if VSTATE["busy"] or VSTATE["data"] is None:
        return
    with vw_img:
        vw_img.clear_output(wait=True)
        plot_single_slice(VSTATE["data"], VSTATE["sel"], VSTATE["var"],
                          vw_orient.value, vw_pos.value, VSTATE["vlim"])
    with vw_wire:
        vw_wire.clear_output(wait=True)
        fig = draw_slice_wireframe(VSTATE["sel"], vw_orient.value, vw_pos.value)
        display(fig); plt.close(fig)


def _draw_domain():
    with vw_domain:
        vw_domain.clear_output(wait=True)
        fig = draw_domain_box(VSTATE["domain"], VSTATE["sel"])
        display(fig); plt.close(fig)


def _set_pos_axis(orient, default="mid"):
    letter, axis = PERP_AXIS[orient]
    n = VSTATE["data"].shape[axis]
    vw_pos.description = letter
    vw_pos.max = max(n - 1, 0)
    vw_pos.value = n // 2 if default == "mid" else min(default, vw_pos.max)


def on_orient_change(change):
    if VSTATE["busy"] or VSTATE["data"] is None:
        return
    VSTATE["busy"] = True; _set_pos_axis(change["new"]); VSTATE["busy"] = False
    _render()


def on_load(_):
    if not vw_file.value:
        with vw_status:
            vw_status.clear_output(wait=True); print("Pick a file first (Refresh if empty).")
        return
    data = np.load(vw_file.value, mmap_mode="r")   # memmap: only touched pages load
    parsed = parse_selection(vw_file.value)
    if parsed:
        var, lvl, sel = parsed
        domain = LEVEL_INFO[lvl]["shape"]
    else:                                   # unrecognised name: box == the cube
        var = os.path.basename(vw_file.value)
        nx, ny, nz = data.shape
        sel = (0, nx, 0, ny, 0, nz)
        domain = data.shape
    VSTATE.update(data=data, sel=sel, var=var, domain=domain,
                  vlim=tuple(np.percentile(np.asarray(data[::4, ::4, ::4]), [2, 98])),
                  busy=True)
    with vw_status:
        vw_status.clear_output(wait=True)
        print(f"Loaded {os.path.basename(vw_file.value)}   shape={data.shape}")
    if 1 in data.shape:                                  # a single plane
        orient = {0: "yz", 1: "xz", 2: "xy"}[list(data.shape).index(1)]
        vw_orient.value = orient; _set_pos_axis(orient, default=0)
        vw_controls.layout.display = "none"
    else:                                                # full 3D cube
        vw_orient.value = "xy"; _set_pos_axis("xy")
        vw_controls.layout.display = ""
    VSTATE["busy"] = False
    _draw_domain()
    _render()


vw_refresh.on_click(on_refresh)
vw_load.on_click(on_load)
vw_orient.observe(on_orient_change, names="value")
vw_pos.observe(_render, names="value")
on_refresh()

viz = widgets.VBox([
    widgets.HTML("<b>Visualize a saved cube</b>"),
    widgets.HBox([vw_file, vw_refresh, vw_load]),
    vw_status, vw_controls,
    widgets.HBox([widgets.VBox([vw_domain, vw_wire]), vw_img]),
], layout=W(padding="12px", max_width="1100px"))
display(viz)